In [2]:
import sys
from sqlalchemy import create_engine, text
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.prompt import Prompt

# --- CONFIGURATION ---
DB_USER = "root"
DB_PASS = "I_amsk77"  # <-- Update with your password
DB_HOST = "127.0.0.1"
DB_PORT = "3306"
DB_NAME = "mini_project"

console = Console()

# Database Connection Test
try:
    db_url = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(db_url)
    with engine.connect() as conn:
        pass
except Exception as e:
    console.print(Panel(
        f"[bold red]Database Connection Error:[/bold red]\n{e}\n\n"
        "[yellow]Fix:[/yellow] Update DB_PASS in hr_cli.py with your correct MySQL password.",
        title="Authentication Failed", border_style="red"
    ))
    sys.exit(1)


# ============================================================================
# 1. CRUD OPERATIONS
# ============================================================================

def create_employee():
    """CREATE: Add a new employee to OLTP and Dim_Employee"""
    console.print(Panel("[bold cyan]CREATE: Add New Employee[/bold cyan]"))
    emp_id = int(Prompt.ask("Enter Employee ID (e.g. 99001)"))
    fname = Prompt.ask("Enter First Name")
    lname = Prompt.ask("Enter Last Name")
    age = int(Prompt.ask("Enter Age", default="28"))
    gender = Prompt.ask("Enter Gender", choices=["Male", "Female", "Non-Binary"], default="Male")
    role = Prompt.ask("Enter Job Role", default="Software Engineer")
    income = float(Prompt.ask("Enter Monthly Income", default="8000.00"))
    dept_id = int(Prompt.ask("Enter Department ID (1=R&D, 2=Sales, 3=HR)", default="1"))

    try:
        with engine.begin() as conn:
            # Insert into OLTP
            conn.execute(text("""
                INSERT INTO Employees (EmployeeID, FirstName, LastName, Age, Gender, DepartmentID, JobRole, MonthlyIncome, IsActive, HireDate)
                VALUES (:emp_id, :fname, :lname, :age, :gender, :dept_id, :role, :income, 1, CURDATE());
            """), {"emp_id": emp_id, "fname": fname, "lname": lname, "age": age, "gender": gender, "dept_id": dept_id, "role": role, "income": income})

            # Insert into OLAP (SCD Type 2 active row)
            conn.execute(text("""
                INSERT INTO Dim_Employee (EmployeeID, Age, Gender, JobRole, MonthlyIncome, IsActive, EffectiveDate, IsCurrent)
                VALUES (:emp_id, :age, :gender, :role, :income, 1, CURDATE(), 1);
            """), {"emp_id": emp_id, "age": age, "gender": gender, "role": role, "income": income})

        console.print(f"[bold green]Employee {emp_id} ({fname} {lname}) successfully created![/bold green]")
    except Exception as e:
        console.print(f"[bold red]Create Failed:[/bold red] {e}")


def read_employee():
    """READ: Search employee by ID"""
    console.print(Panel("[bold cyan]READ: Lookup Employee[/bold cyan]"))
    emp_id = int(Prompt.ask("Enter Employee ID"))

    query = """
    SELECT e.EmployeeID, e.FirstName, e.LastName, d.DepartmentName, e.JobRole, e.MonthlyIncome, e.IsActive
    FROM Employees e
    LEFT JOIN Departments d ON e.DepartmentID = d.DepartmentID
    WHERE e.EmployeeID = :emp_id;
    """
    with engine.connect() as conn:
        row = conn.execute(text(query), {"emp_id": emp_id}).fetchone()

    if row:
        table = Table(title=f"Employee Record: {emp_id}", header_style="bold green")
        table.add_column("ID", justify="center")
        table.add_column("Name")
        table.add_column("Department")
        table.add_column("Role")
        table.add_column("Monthly Income", justify="right")
        table.add_column("Status", justify="center")
        
        status = "Active" if row[6] == 1 else "Inactive"
        table.add_row(str(row[0]), f"{row[1]} {row[2]}", str(row[3]), str(row[4]), f"${row[5]:,.2f}", status)
        console.print(table)
    else:
        console.print(f"[bold yellow]No employee found with ID {emp_id}[/bold yellow]")


def update_employee():
    """UPDATE: Modify Job Role and Salary (Triggers SCD2 in DW)"""
    console.print(Panel("[bold cyan]UPDATE: Modify Employee Role & Salary[/bold cyan]"))
    emp_id = int(Prompt.ask("Enter Employee ID to update"))
    new_role = Prompt.ask("Enter New Job Role")
    new_income = float(Prompt.ask("Enter New Monthly Income"))

    try:
        with engine.begin() as conn:
            # 1. Update OLTP
            conn.execute(text("""
                UPDATE Employees 
                SET JobRole = :role, MonthlyIncome = :income 
                WHERE EmployeeID = :emp_id;
            """), {"role": new_role, "income": new_income, "emp_id": emp_id})

            # 2. Update SCD Type 2 in Data Warehouse
            conn.execute(text("""
                UPDATE Dim_Employee 
                SET ExpirationDate = CURDATE(), IsCurrent = 0 
                WHERE EmployeeID = :emp_id AND IsCurrent = 1;
            """), {"emp_id": emp_id})

            conn.execute(text("""
                INSERT INTO Dim_Employee (EmployeeID, JobRole, MonthlyIncome, IsActive, EffectiveDate, IsCurrent)
                VALUES (:emp_id, :role, :income, 1, CURDATE(), 1);
            """), {"emp_id": emp_id, "role": new_role, "income": new_income})

        console.print(f"[bold green]Employee {emp_id} updated. SCD2 version history preserved![/bold green]")
    except Exception as e:
        console.print(f"[bold red]Update Failed:[/bold red] {e}")


def delete_employee():
    """DELETE: Soft delete an employee"""
    console.print(Panel("[bold cyan]DELETE: Soft Delete Employee[/bold cyan]"))
    emp_id = int(Prompt.ask("Enter Employee ID to deactivate"))

    try:
        with engine.begin() as conn:
            # Soft delete in OLTP
            conn.execute(text("UPDATE Employees SET IsActive = 0 WHERE EmployeeID = :emp_id;"), {"emp_id": emp_id})
            # Soft delete in OLAP
            conn.execute(text("""
                UPDATE Dim_Employee 
                SET IsActive = 0, Attrition = 'Yes', ExpirationDate = CURDATE(), IsCurrent = 0 
                WHERE EmployeeID = :emp_id AND IsCurrent = 1;
            """), {"emp_id": emp_id})

        console.print(f"[bold yellow]Employee {emp_id} deactivated successfully.[/bold yellow]")
    except Exception as e:
        console.print(f"[bold red]Delete Failed:[/bold red] {e}")


# ============================================================================
# 2. KPI ANALYTICS ENGINE
# ============================================================================

def get_yoy_performance():
    """KPI 1: Year-over-Year Performance Trends"""
    query = """
    SELECT d.Year, COUNT(f.ReviewID), ROUND(AVG(f.PerformanceRating), 2), ROUND(AVG(f.JobSatisfaction), 2)
    FROM Fact_PerformanceReviews f 
    JOIN Dim_Date d ON f.ReviewDateKey = d.DateKey
    GROUP BY d.Year ORDER BY d.Year ASC;
    """
    with engine.connect() as conn:
        rows = conn.execute(text(query)).fetchall()

    table = Table(title="KPI 1: Year-over-Year Performance Trends", header_style="bold cyan")
    table.add_column("Year", justify="center")
    table.add_column("Total Reviews", justify="right")
    table.add_column("Avg Rating", justify="right")
    table.add_column("Avg Job Sat.", justify="right")

    for r in rows:
        table.add_row(str(r[0]), f"{r[1]:,}", str(r[2]), str(r[3]))
    console.print(table)


def get_top_performers():
    """KPI 2: Top Performers by Dept using DENSE_RANK()"""
    query = """
    WITH Ranked AS (
        SELECT dept.DepartmentName, emp.EmployeeID, emp.JobRole, f.PerformanceRating,
               DENSE_RANK() OVER (PARTITION BY dept.DepartmentName ORDER BY f.PerformanceRating DESC) AS PerfRank
        FROM Fact_PerformanceReviews f
        JOIN Dim_Employee emp ON f.EmployeeSK = emp.EmployeeSK
        JOIN Dim_Department dept ON f.DepartmentSK = dept.DepartmentSK
        WHERE emp.IsCurrent = 1
    )
    SELECT DepartmentName, PerfRank, EmployeeID, JobRole, PerformanceRating 
    FROM Ranked WHERE PerfRank <= 3 ORDER BY DepartmentName, PerfRank;
    """
    with engine.connect() as conn:
        rows = conn.execute(text(query)).fetchall()

    table = Table(title="KPI 2: Top 3 Performers by Department (DENSE_RANK)", header_style="bold green")
    table.add_column("Department")
    table.add_column("Rank", justify="center", style="yellow")
    table.add_column("Emp ID", justify="center")
    table.add_column("Role")
    table.add_column("Rating", justify="right")

    for r in rows:
        table.add_row(str(r[0]), str(r[1]), str(r[2]), str(r[3]), str(r[4]))
    console.print(table)


def get_attrition_and_bottlenecks():
    """KPI 3: Attrition Risk & Project Bottlenecks"""
    query = """
    SELECT dept.DepartmentName, COUNT(emp.EmployeeSK) AS TotalEmployees,
           SUM(CASE WHEN emp.Attrition = 'Yes' THEN 1 ELSE 0 END) AS AttritionCount,
           ROUND((SUM(CASE WHEN emp.Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(emp.EmployeeSK)) * 100, 2) AS AttritionRatePct,
           ROUND(AVG(f.AllocatedHours), 1) AS AvgWeeklyHours
    FROM Fact_PerformanceReviews f
    JOIN Dim_Employee emp ON f.EmployeeSK = emp.EmployeeSK
    JOIN Dim_Department dept ON f.DepartmentSK = dept.DepartmentSK
    WHERE emp.IsCurrent = 1 
    GROUP BY dept.DepartmentName ORDER BY AttritionRatePct DESC;
    """
    with engine.connect() as conn:
        rows = conn.execute(text(query)).fetchall()

    table = Table(title="KPI 3: Attrition Risk & Workload Bottlenecks", header_style="bold red")
    table.add_column("Department")
    table.add_column("Headcount", justify="right")
    table.add_column("Attritions", justify="right")
    table.add_column("Attrition Rate (%)", justify="right", style="bold red")
    table.add_column("Avg Weekly Hours", justify="right")

    for r in rows:
        table.add_row(str(r[0]), f"{r[1]:,}", str(r[2]), f"{r[3]}%", str(r[4]))
    console.print(table)


# ============================================================================
# 3. SCD TYPE 2 VERIFICATION ENGINE
# ============================================================================

def test_scd2_history():
    """TEST SCD: Check version lineage for any employee"""
    console.print(Panel("[bold magenta]SCD TYPE 2 LINEAGE INSPECTOR[/bold magenta]"))
    emp_id = int(Prompt.ask("Enter Employee ID to inspect version history", default="1"))

    query = """
    SELECT EmployeeSK, EmployeeID, JobRole, MonthlyIncome, EffectiveDate, ExpirationDate, IsCurrent
    FROM Dim_Employee
    WHERE EmployeeID = :emp_id
    ORDER BY EmployeeSK ASC;
    """
    with engine.connect() as conn:
        rows = conn.execute(text(query), {"emp_id": emp_id}).fetchall()

    if rows:
        table = Table(title=f"Dim_Employee Version Lineage for ID: {emp_id}", header_style="bold magenta")
        table.add_column("Surrogate SK", justify="center")
        table.add_column("Natural ID", justify="center")
        table.add_column("Job Role")
        table.add_column("Monthly Income", justify="right")
        table.add_column("Effective Date", justify="center")
        table.add_column("Expiration Date", justify="center")
        table.add_column("IsCurrent", justify="center", style="bold yellow")

        for r in rows:
            exp_date = str(r[5]) if r[5] else "NULL (Active)"
            curr_flag = "1 (True)" if r[6] == 1 else "0 (False)"
            table.add_row(str(r[0]), str(r[1]), str(r[2]), f"${r[3]:,.2f}", str(r[4]), exp_date, curr_flag)
        
        console.print(table)
    else:
        console.print(f"[bold yellow]No lineage records found in Dim_Employee for ID {emp_id}[/bold yellow]")


# ============================================================================
# 4. MAIN INTERACTIVE MENU
# ============================================================================

def main():
    while True:
        console.clear()
        console.print(Panel.fit(
            "[bold white]Data Warehouse Management & Analytics CLI[/bold white]\n"
            "[dim]Select an option to execute[/dim]",
            title="[bold yellow]MINI_PROJECT ENGINE[/bold yellow]", border_style="blue"
        ))
        
        console.print("[bold cyan]--- CRUD Operations ---[/bold cyan]")
        console.print(" [1] Create (Add Employee)")
        console.print(" [2] Read (Lookup Employee)")
        console.print(" [3] Update (Modify Role & Salary - Triggers SCD2)")
        console.print(" [4] Delete (Soft Delete Employee)")
        console.print("\n[bold cyan]--- KPI Analytics ---[/bold cyan]")
        console.print(" [5] KPI 1: Year-over-Year Performance Trends")
        console.print(" [6] KPI 2: Top Performers by Dept (DENSE_RANK)")
        console.print(" [7] KPI 3: Attrition Risk & Project Bottlenecks")
        console.print("\n[bold cyan]--- Testing & Auditing ---[/bold cyan]")
        console.print(" [8] Test SCD Type 2 Lineage (Inspect Version Rows)")
        console.print(" [9] Exit CLI")
        
        choice = Prompt.ask("\nEnter choice", choices=["1", "2", "3", "4", "5", "6", "7", "8", "9"], default="1")

        if choice == "1": create_employee()
        elif choice == "2": read_employee()
        elif choice == "3": update_employee()
        elif choice == "4": delete_employee()
        elif choice == "5": get_yoy_performance()
        elif choice == "6": get_top_performers()
        elif choice == "7": get_attrition_and_bottlenecks()
        elif choice == "8": test_scd2_history()
        elif choice == "9":
            console.print("\n[bold green]Exiting CLI. Have a great day![/bold green]\n")
            sys.exit(0)

        Prompt.ask("\n[dim]Press Enter to return to main menu...[/dim]")


if __name__ == "__main__":
    main()

╭─────────── MINI_PROJECT ENGINE ───────────╮
│ Data Warehouse Management & Analytics CLI │
│ Select an option to execute               │
╰───────────────────────────────────────────╯

--- CRUD Operations ---

[1] Create (Add Employee)

[2] Read (Lookup Employee)

[3] Update (Modify Role & Salary - Triggers SCD2)

...

Please select one of the available options

Enter choice [1/2/3/4/5/6/7/8/9] (1):

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ CREATE: Add New Employee                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter Employee ID (e.g. 99001):

ValueError: invalid literal for int() with base 10: ''